In [ ]:
import os
import numpy as np
import pandas as pd


import pickle
from pathlib import Path

In [ ]:
results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-01-27_curate_controls_for_PheWAS_study"
results1 = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-11-13_get_conditions_of_cohorts"

data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-01-27_curate_controls_for_PheWAS_study"
data2 ="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-11-13_get_cohort_statistic"

scratch="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2026-01-27_curate_controls_for_PheWAS_study"



In [ ]:
#1 upload ALL genomics data having patients



In [ ]:
# genomics function 

#This query represents dataset "Dem data for has genomic data cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8

def get_genomics_data_cohort():
    
    dataset_43111160_person_sql = """
        SELECT
            person.person_id,
            person.gender_concept_id,
            p_gender_concept.concept_name as gender,
            person.birth_datetime as date_of_birth,
            person.race_concept_id,
            p_race_concept.concept_name as race,
            person.ethnicity_concept_id,
            p_ethnicity_concept.concept_name as ethnicity,
            person.sex_at_birth_concept_id,
            p_sex_at_birth_concept.concept_name as sex_at_birth,
            person.self_reported_category_concept_id,
            p_self_reported_category_concept.concept_name as self_reported_category 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
                ON person.gender_concept_id = p_gender_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
                ON person.race_concept_id = p_race_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
                ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
                ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
                ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
        WHERE
            person.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 
                UNION
                DISTINCT SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_lr_whole_genome_variant = 1 
                UNION
                DISTINCT SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_array_data = 1 ) )"""

    genomics_person_df = pd.read_gbq(
        dataset_43111160_person_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook")

    return genomics_person_df

In [ ]:
#2 upload phecode and phecodeX donor files and melt into long format, and filter for patients who have yes in phecodes column
## filter donors based on if their person_id is found in the genomics cohort df


In [ ]:
def get_phecode_donor_file(genomics_cohort):
    # ---------- File 1 ----------
    wide1 = pd.read_csv(f"{data}/mcc2_phecode_table.csv", dtype={"person_id": str})

    # melt into long format
    donor_codes = wide1.melt(
        id_vars="person_id",
        var_name="phecode",
        value_name="has_code"
    )

    # keep only codes that are present (True / 1)
    donor_codes = donor_codes[donor_codes["has_code"].astype(bool)].drop(columns="has_code")
    
    
    
    #filter by has genomics data based on person_id found in genomics cohort
    
    genomics_cohort["person_id"] = genomics_cohort["person_id"].astype(str)
    donor_codes["person_id"] = donor_codes["person_id"].astype(str)
    
    
    genomics_cohort_ids = genomics_cohort["person_id"].unique()
    donor_codes = donor_codes[donor_codes["person_id"].isin(genomics_cohort_ids)]
    
    
     #set column type as str
    donor_codes['phecode'] = donor_codes['phecode'].astype(str)
    donor_codes = donor_codes.drop_duplicates()
    
    
    
    
    return donor_codes


In [ ]:
def get_phecode_donor_file2():
    # ---------- File 1 ----------
    wide1 = pd.read_csv(f"{data}/mcc2_phecode_table.csv", dtype={"person_id": str})

    # melt into long format
    donor_codes = wide1.melt(
        id_vars="person_id",
        var_name="phecode",
        value_name="has_code"
    )

    # keep only codes that are present (True / 1)
    donor_codes = donor_codes[donor_codes["has_code"].astype(bool)].drop(columns="has_code")
    
    
    
    
    donor_codes["person_id"] = donor_codes["person_id"].astype(str)
    
    
    
    
     #set column type as str
    donor_codes['phecode'] = donor_codes['phecode'].astype(str)
    donor_codes = donor_codes.drop_duplicates()
    
    
    
    
    return donor_codes


In [ ]:
def get_phecodeX_donor_file(genomics_cohort):
    
    wide2 = pd.read_csv(f"{data}/mcc2_phecodex_table.csv", dtype={"person_id": str}, usecols=lambda c: c != "sex")

    # melt into long format
    donor_codes = wide2.melt(
        id_vars="person_id",
        var_name="phecode",
        value_name="has_code"
    )
    
    # keep only codes that are present (True / 1)
    donor_codes = donor_codes[donor_codes["has_code"].astype(bool)].drop(columns="has_code")
    
    #filter by has genomics data based on person_id found in genomics cohort
    
    genomics_cohort["person_id"] = genomics_cohort["person_id"].astype(str)
    donor_codes["person_id"] = donor_codes["person_id"].astype(str)
    
    genomics_cohort_ids = genomics_cohort["person_id"].unique()

    donor_codes = donor_codes[donor_codes["person_id"].isin(genomics_cohort_ids)]
    
     #set column type as str
    donor_codes['phecode'] = donor_codes['phecode'].astype(str)
    donor_codes = donor_codes.drop_duplicates()

    return donor_codes

In [ ]:
#3 upload phecode decscription, group, and ICD mapping files and merge



In [ ]:
def import_map_files():
   
    
    
    phe_icd= pd.read_csv(f"{data}/expanded_phecode.csv")
    
    phe = pd.read_csv(f"{data}/phecode_info.csv")
    
    phex = pd.read_csv(f"{data}/phecodex_info.csv")
    
    phex = phex.rename({"phecodex": "phecode"}, axis="columns")
    
    phex_icd = pd.read_csv(f"{data}/updated_phecodex_map.csv")
    
    
    return phe, phe_icd, phex, phex_icd

In [ ]:
def merge_phecodes_with_phenotype(phe, phe_icd):
    
    final = pd.merge(phe, phe_icd, how = "inner")
    
    return final

In [ ]:
#4 using phecode/phecodex mapping file to count number of donors per phegroup



In [ ]:
# all B ICDs (phecode & phecodex), one row per phecode–ICD
# join B ICDs to donors
def get_phecode_donor_icd_phegroup_map(phecode_donor_map, phe_codes):

    phe_donors = phe_codes.merge(
        phecode_donor_map,
        on="phecode",
        how="left"
    ).dropna(subset=["person_id"])
    
    

    # one row per (B, donor)
    phe_donors_unique = phe_donors[["phecode", "person_id"]].drop_duplicates()

    # count donors per phecode
    b_counts = (
        phe_donors_unique
        .groupby(["phecode"])["person_id"]
        .nunique()
        .rename("sample_count")
        .reset_index()
    )
    
    return phe_donors, b_counts

In [ ]:
#function calls


genomics_cohort = get_genomics_data_cohort()
'''
#phe_donors = get_phecode_donor_file(genomics_cohort)
#phex_donors = get_phecodeX_donor_file(genomics_cohort)


phe, phe_icd, phex, phex_icd = import_map_files()
phe_map = merge_phecodes_with_phenotype(phe, phe_icd)
phex_map = merge_phecodes_with_phenotype(phex, phex_icd)
'''


#phe_donor_map, phe_donor_count = get_phecode_donor_icd_phegroup_map(phe_donors, phe_map)
#phex_donor_map, phex_donor_count = get_phecode_donor_icd_phegroup_map(phex_donors, phex_map)

In [ ]:
phe_donors = get_phecode_donor_file(genomics_cohort)

In [ ]:
phex_donors = get_phecodeX_donor_file(genomics_cohort)

In [ ]:
#visualize

phe_donors.head()

In [ ]:
raw_donor = get_phecode_donor_file2()

In [ ]:
viral_hep_c_donors = set(phe_donors[phe_donors['phecode'] == '070.3']['person_id'])

In [ ]:
len(viral_hep_c_donors)

In [ ]:
donors_with_genomic = set(genomics_cohort['person_id'])

In [ ]:
len(donors_with_genomic)

In [ ]:
genomics_cohort

In [ ]:
genomics_cohort

In [ ]:
viral_hep_c_and_genomic = viral_hep_c_donors.intersection(donors_with_genomic)

In [ ]:
len(viral_hep_c_and_genomic)